# verses

> a printed verse number read, verse runs mended and renumbered, verses assembled into chapters, Devanagari akṣaras repaired

In [ ]:
#| default_exp verses

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re
from copy import deepcopy
from difflib import SequenceMatcher
from fastcore.all import AttrDict, L

A verse is `{n, text, page, heading}`. A page is `{page, blocks}`, where each block is `{kind, n, text}`: what `vishalakshi.read_doc` keeps from a vision model's reading. A page may keep its blocks under `items` instead, as vedicreader's readings do; `blocks` finds them either way. `kind` reads a block's kind through `KINDS`, so a reader's `head` is a `heading`. Every function that puts a number right notes what it did in `probs`, a list the caller passes in, for a proofreader to check.

In [ ]:
#| export
KINDS = dict(head='heading')
def kind(b) -> str|None:
    "A block's kind, an alias in `KINDS` read as the kind it names: `head` is `heading`."
    k = b.get('kind'); return KINDS.get(k, k)

def _bk(p): return 'items' if 'items' in p and 'blocks' not in p else 'blocks'
def blocks(p) -> list:
    "A page's blocks, kept under `blocks` or, as vedicreader keeps them, `items`."
    if p.get(k := _bk(p)) is None: p[k] = []
    return p[k]

In [ ]:
test_eq(blocks(dict(page=1, items=[1])), [1]); test_eq(blocks(dict(page=1, blocks=[2], items=[1])), [2]); test_eq(blocks(dict(page=1)), [])
test_eq(kind(dict(kind='head')), 'heading'); test_eq(kind(dict(kind='verse')), 'verse'); test_eq(kind({}), None)

## Verse numbers

`vnum` reads a printed verse number in ASCII, Devanagari or Tamil digits. Tamil numbers written with the old signs ௰ ten, ௱ hundred and ௲ thousand compose (௬௱௨ is 602). Of two numbers printed together it takes the `last` and notes it: `२६॥२७` is two verses read as one. `pick='first'` suits a print that sets a running number before a count within its part (`२६॥ १`).

In [ ]:
#| export
_DIG = str.maketrans('०१२३४५६७८९௦௧௨௩௪௫௬௭௮௯', '01234567890123456789')
_SIGNS = {'௰': 10, '௱': 100, '௲': 1000}
def _tamil(s):
    "A run of digits and the old Tamil signs ௰ ten, ௱ hundred, ௲ thousand as an int: ௬௱௨ is 602, ௨௱௩ 203."
    tot, d = 0, None
    for c in s:
        if c.isdecimal(): d = (d or 0) * 10 + int(c)
        else: tot += (d or 1) * _SIGNS[c]; d = None
    return tot + (d or 0)

def vnum(n, page=None, probs:list=None, pick:str='last') -> int|None:
    "A printed verse number as an int, or None when it has no digits; of several printed together, the `last` or the `first`."
    xs = [_tamil(x) for x in re.findall(r'[\d௰௱௲]+', str('' if n is None else n).translate(_DIG))]
    if len(xs) > 1 and probs is not None: probs.append(f'{page}: verses {"-".join(map(str, xs))} printed as one, numbered {xs[-1 if pick == "last" else 0]}')
    return (xs[-1] if pick == 'last' else xs[0]) if xs else None

In [ ]:
_p = []
test_eq(vnum('॥ २६ ॥'), 26); test_eq(vnum('(௪௦)'), 40); test_eq(vnum(12), 12); test_eq(vnum(None), None)
test_eq(vnum('२६॥२७', 'p5', _p), 27); test_eq(_p, ['p5: verses 26-27 printed as one, numbered 27'])
test_eq(vnum('௬௱௨'), 602); test_eq(vnum('௨௱௩'), 203); test_eq(vnum('௰'), 10)
test_eq(vnum('२६॥ १', pick='first'), 26)   # a running number, then the count within its part

## Mending a run

`mend` puts right two slips a reader makes, where the neighbours prove them: a number read twice where the one before is missing (148, 150, 150: the first is 149), and two verses read as one where a number is missing between them, split after the inner line that closes a verse. `pair`, for a book of two-line verses whose halves both close on a daṇḍa, cuts where the two parts are nearest in length. `renumber` mends a long chapter's numbers: a run read at a constant offset that the sequence then returns from (53-58 read 63-68, then 59), and a number read without its leading digit (101 as 1).

In [ ]:
#| export
_END = re.compile(r'[.।॥|]\s*$')
def _ln(vs): return next((v['n'] for v in reversed(vs) if v['n'] is not None), None)
def mend(vs, probs:list, pair:bool=False) -> list:
    "Verses with a doubled number and two verses read as one put right, where their neighbours prove it."
    out = []
    for i, v in enumerate(vs):
        if v['n'] is None: out.append(v); continue
        n, prev = v['n'], _ln(out) or 0
        nxt = vs[i + 1]['n'] if i + 1 < len(vs) else None
        if n == nxt and prev == n - 2:
            out.append(dict(v, n=n - 1)); probs.append(f"{v['page']}: first of two {n}s taken as {n - 1}"); continue
        ls = v['text'].split('\n')
        cut = [j for j, l in enumerate(ls[:-1]) if _END.search(l) and j >= (1 if pair else 2)]
        if prev == n - 2 and cut:
            j = (min(cut, key=lambda c: abs(len(''.join(ls[:c + 1])) - len(''.join(ls[c + 1:])))) if pair else cut[-1]) + 1
            out += [dict(v, n=n - 1, text='\n'.join(ls[:j])), dict(v, text='\n'.join(ls[j:]), heading=None)]
            probs.append(f"{v['page']}: {n} split after its line {j}, the first part taken as {n - 1}"); continue
        out.append(v)
    return out

def renumber(vs, where, probs:list) -> list:
    "Verse numbers a long chapter's print makes hard to read, put right from their neighbours; an unnumbered verse stays where it is."
    ix, out = [i for i, v in enumerate(vs) if v['n'] is not None], [dict(v) for v in vs]
    for i, v in zip(ix, _renum([vs[i] for i in ix], where, probs)): out[i] = v
    return out

def _renum(vs, where, probs):
    vs = [dict(v) for v in vs]
    for i in range(1, len(vs)):
        want = vs[i - 1]['n'] + 1; d = vs[i]['n'] - want
        if not d: continue
        j = i
        while j < len(vs) and j - i < 15 and vs[j]['n'] - d == want + (j - i): j += 1
        if i < j < len(vs) and vs[j]['n'] == want + (j - i):
            for k in range(i, j): probs.append(f"{where}: verse read as {vs[k]['n']} numbered {vs[k]['n'] - d} ({vs[k]['page']})"); vs[k]['n'] -= d
    out = []
    for i, v in enumerate(vs):
        n = v['n']; want = out[-1]['n'] + 1 if out else n
        nxt = vs[i + 1]['n'] if i + 1 < len(vs) else None
        k = 10 ** len(str(n))
        if n != want and (nxt == want + 1 or (n < want and want % k == n % k)):
            probs.append(f"{where}: verse read as {n} numbered {want} ({v['page']})"); n = want
        out.append(dict(v, n=n))
    return out

In [ ]:
V = lambda n, t='a ।\nb ॥', pg=1: dict(n=n, text=t, page=pg, heading=None)
_p = []
test_eq([v['n'] for v in mend([V(148), V(150), V(150), V(151)], _p)], [148, 149, 150, 151]); test_eq(len(_p), 1)
_p = []
_m = mend([V(1), V(3, 'a\nb\nc ॥\nd\ne ॥')], _p)
test_eq([(v['n'], v['text']) for v in _m], [(1, 'a ।\nb ॥'), (2, 'a\nb\nc ॥'), (3, 'd\ne ॥')])
_p = []
test_eq([v['n'] for v in renumber([V(n) for n in (52, 63, 64, 65, 56, 57)], 'ch 3', _p)], [52, 53, 54, 55, 56, 57]); test_eq(len(_p), 3)
test_eq([v['n'] for v in renumber([V(n) for n in (99, 100, 1, 102)], 'ch 9', [])], [99, 100, 101, 102])
test_eq([v['n'] for v in renumber([V(None), V(99), V(100), V(1), V(102)], 'ch 9', [])], [None, 99, 100, 101, 102])   # an unnumbered opener stays put
test_eq([v['n'] for v in mend([V(None), V(1), V(3, 'a\nb\nc ॥\nd\ne ॥')], [])], [None, 1, 2, 3])

## Chapters

`assemble` gathers verses from pages in order. An unnumbered fragment carries onto the next numbered verse (a verse begun on one page and closed on the next). A heading names the chapter it opens; a verse numbered 1 after a higher number starts a new chapter; a colophon closes one, taking an unnumbered verse just before it as the chapter's last. The same verse read twice (a spread photographed twice) is kept once. Each chapter's run is then mended. Only `kinds` count as verse (`verse`, or a block with no kind); `kinds=None` takes every block but headings and colophons, for a reading that labels verse text by its own kinds. A commentary a reading labels by its own kind (an urai, a gloss, a bhāṣya) belongs to its verse: name the kinds in `notes` and each goes into its verse's `notes`, under its kind. Some prints number the commentary, not the verse (the Vidhānamālai sets the number at the urai's end): `number_from_note` gives the verse that number. It returns the chapters and the problems seen. With `cont`, a block a page break cut is first joined to its start (`join_cont`): a reader marks the continuing block `cont`, and a word hyphenated at the break is rejoined. A reader's `head` is a heading, and pages may keep their blocks under `items`. With `openers`, an unnumbered block closed by ॥ (a maṅgala, an opening verse) stays a verse of its own, `n` None, in the chapter of the numbered verse after it, where by default it joins that verse; `mend` and `renumber` leave such a verse where it is.

`join_cont` is that join on its own: the continuing block's text goes onto the last block of the page before when the two are of one kind, joined by a line break for a verse and a space otherwise, and its printed number goes with it.

In [ ]:
#| export
def join_cont(pages) -> list:
    "Pages with a block a page break cut (`cont` on the next page's first block) joined to its start, a word hyphenated at the break rejoined."
    last = None
    for p in pages:
        bl = blocks(p)
        if bl and last is not None and bl[0].get('cont') and kind(bl[0]) == kind(last):
            b = bl.pop(0); a, t = (last.get('text') or '').rstrip(), (b.get('text') or '').strip()
            last['text'] = a[:-1] + t if a.endswith('-') else a + ('\n' if kind(b) == 'verse' else ' ') + t
            if b.get('n') is not None: last['n'] = b['n']
        if bl: last = bl[-1]
    return pages

In [ ]:
_jc = [dict(page=1, items=[dict(kind='prose', text='a long com-'), dict(kind='verse', text='क ।')]),
       dict(page=2, items=[dict(kind='verse', text='ख ॥२॥', n=2, cont=True), dict(kind='prose', text='next')]),
       dict(page=3, items=[dict(kind='prose', text='one', cont=True)])]
test_eq([(b['kind'], b['text'], b.get('n')) for p in join_cont(_jc) for b in p['items']], [('prose', 'a long com-', None), ('verse', 'क ।\nख ॥२॥', 2), ('prose', 'next one', None)])
test_eq(join_cont([dict(page=1, blocks=[dict(kind='prose', text='com-')]), dict(page=2, blocks=[dict(kind='prose', text='mentary', cont=True)])])[0]['blocks'][0]['text'], 'commentary')

In [ ]:
#| export
def _alike(a, b, at=.8): return SequenceMatcher(None, a, b).ratio() >= at

def _take(n, t, pg, buf, have, probs):
    "The verse `{n, text, page}` a block closes; None when it has no number and joins `buf`, False when it repeats one of `have`."
    if (n := vnum(n, pg, probs) if n is not None else None) is None:
        if not (buf and _alike(buf[-1], t)): buf.append(t)
        return None
    if any(v['n'] == n and _alike(v['text'].split('\n')[-1], t.split('\n')[-1], .6) for v in have): return False
    v = dict(n=n, text='\n'.join(buf + [t]), page=pg); buf.clear()
    return v

def assemble(pages, pair:bool=False, kinds=('verse', None), notes=(), number_from_note:bool=False, cont:bool=False, openers:bool=False) -> tuple:
    """Chapters `[{heading, colophon, verses}]` from pages of blocks, and the problems seen. `kinds` None takes every block but headings and
    colophons as verse. A block of a kind in `notes` (`urai`, `gloss`, `bhashya`) goes into its verse's `notes`; with `number_from_note`, a
    numbered note gives the unnumbered verse before it that number. `cont` joins a block cut by a page break first (`join_cont`). With
    `openers`, an unnumbered block closed by ॥ (a maṅgala) is a verse of its own, `n` None, in the chapter of the verse after it."""
    if cont: pages = join_cont(deepcopy(list(pages)))
    chs, cur, buf, probs, hd, pend, opn = [], None, [], [], None, {}, []
    new = lambda h=None: dict(heading=h, colophon=None, verses=[])
    def add(n, text, pg):
        nonlocal cur, hd, pend
        if cur is None or (cur['verses'] and n == 1 and cur['verses'][-1]['n'] > 1):
            if cur and cur['verses']: chs.append(cur)
            cur = new()
        if cur['verses'] and n != cur['verses'][-1]['n'] + 1: probs.append(f"{pg}: verse {n} after {cur['verses'][-1]['n']}")
        v = dict(n=n, text=text, page=pg, heading=hd)
        if pend: v['notes'], pend = pend, {}
        cur['verses'] += opn + [v]; opn.clear(); hd = None
        return v
    for p in pages:
        for b in p.get(_bk(p)) or []:
            k, t, n, pg = kind(b), b.get('text') or '', b.get('n'), p.get('page')   # text as read: a leading break or an empty block is the reader's
            if k in notes:
                m = vnum(n, pg, probs) if n is not None else None
                if number_from_note and buf and m is not None:
                    add(m, '\n'.join(buf), pg); buf = []; probs.append(f'{pg}: verse numbered {m} from its {k}')
                last = None if buf else opn[-1] if opn else cur['verses'][-1] if cur and cur['verses'] else None
                (last.setdefault('notes', {}) if last else pend).setdefault(k, []).append(t)
                continue
            if k == 'start':   # a chapter opens here: `repairs.cut_by_plan`
                if cur and cur['verses']: chs.append(cur)
                cur = None; continue
            if k == 'heading':
                hd = t
                if cur is None: cur = new(t)
                elif not cur['heading']: cur['heading'] = t
                continue
            if k == 'colophon':
                if buf and cur and cur['verses']:
                    n = cur['verses'][-1]['n'] + 1
                    cur['verses'].append(dict(n=n, text='\n'.join(buf), page=pg, heading=None)); buf = []
                    probs.append(f'{pg}: unnumbered verse before the colophon taken as {n}')
                if opn: cur = cur or new(); cur['verses'] += opn; opn.clear()
                (cur or new())['colophon'] = t
                if cur: chs.append(cur); cur = None
                continue
            if kinds is not None and k not in kinds: continue
            if openers and vnum(n) is None and t.rstrip().endswith('॥'):
                opn.append(dict(n=None, text='\n'.join(buf + [t]), page=pg, heading=hd)); buf.clear(); hd = None; continue
            if (v := _take(n, t, pg, buf, cur['verses'] if cur else [], probs)) is False: hd = None
            elif v: add(v['n'], v['text'], pg)
    if opn: cur = cur or new(); cur['verses'] += opn
    if cur and cur['verses']: chs.append(cur)
    if buf: probs.append(f'unfinished verse at the end: {buf[-1][:40]}')
    for c in chs: c['verses'] = mend(c['verses'], probs, pair)
    return chs, probs

def check_counts(chs, counts) -> list:
    "Where the chapters' verse counts differ from the counts a plan expects."
    out = [] if len(chs) == len(counts) else [f'{len(chs)} chapters, the plan has {len(counts)}']
    return out + [f"chapter {i + 1}: {len(c['verses'])} verses, the plan has {k}" for i, (c, k) in enumerate(zip(chs, counts)) if len(c['verses']) != k]

In [ ]:
B = lambda kind, text, n=None: dict(kind=kind, text=text, n=n)
_pages = [dict(page=0, blocks=[B('heading', 'अथ प्रथमोऽध्यायः'), B('verse', 'क ।\nख ॥१॥', '१'), B('verse', 'ग ।', None)]),
          dict(page=1, blocks=[B('verse', 'घ ॥२॥', '॥२॥'), B('verse', 'ङ ॥३॥', 3), B('colophon', 'इति प्रथमोऽध्यायः')]),
          dict(page=2, blocks=[B('verse', 'च ॥१॥', 1), B('verse', 'छ ॥२॥', 2), B('verse', 'छ ॥२॥', 2), B('prose', 'an English note')])]
_chs, _pr = assemble(_pages)
test_eq(len(_chs), 2); test_eq(_chs[0]['heading'], 'अथ प्रथमोऽध्यायः'); test_eq(_chs[0]['colophon'], 'इति प्रथमोऽध्यायः')
test_eq([(v['n'], v['text']) for v in _chs[0]['verses']], [(1, 'क ।\nख ॥१॥'), (2, 'ग ।\nघ ॥२॥'), (3, 'ङ ॥३॥')])
test_eq([v['n'] for v in _chs[1]['verses']], [1, 2]); test_eq(_pr, [])
test_eq(check_counts(_chs, [3, 2]), []); test_eq(check_counts(_chs, [3, 3]), ['chapter 2: 2 verses, the plan has 3'])

In [ ]:
_u = [dict(page=0, blocks=[B('verse', 'பா', 1), B('urai', 'உரை', 1)])]
test_eq(len(assemble(_u)[0][0]['verses']), 1); test_eq([v['text'] for v in assemble(_u, kinds=None)[0][0]['verses']], ['பா', 'உரை'])   # every kind but headings and colophons

In [ ]:
_vm = [dict(page=0, blocks=[B('verse', 'பா ஒன்று', None), B('urai', 'உரை ஒன்று', 1), B('verse', 'பா இரண்டு', None), B('urai', 'உரை இரண்டு', 2)])]
_c, _p = assemble(_vm, notes=('urai',), number_from_note=True)
test_eq([(v['n'], v['text'], v['notes']['urai']) for v in _c[0]['verses']], [(1, 'பா ஒன்று', ['உரை ஒன்று']), (2, 'பா இரண்டு', ['உரை இரண்டு'])])
_g = [dict(page=0, blocks=[B('verse', 'a ॥१॥', 1), B('gloss', 'its gloss'), B('verse', 'b ॥२॥', 2)])]
test_eq(assemble(_g, notes=('gloss',))[0][0]['verses'][0]['notes'], {'gloss': ['its gloss']})
test_eq('notes' in assemble(_g)[0][0]['verses'][0], False)   # without `notes` a gloss is not a verse and is not kept

In [ ]:
_it = [dict(page=0, items=[B('head', 'अथ प्रथमोऽध्यायः'), B('verse', 'क ।\nख ॥१॥', 1), dict(kind='verse', text='ग ।', open=True)]),
       dict(page=1, items=[dict(kind='verse', text='घ ॥२॥', n=2, cont=True)])]
_c, _p = assemble(_it, cont=True)
test_eq((_c[0]['heading'], [(v['n'], v['text']) for v in _c[0]['verses']]), ('अथ प्रथमोऽध्यायः', [(1, 'क ।\nख ॥१॥'), (2, 'ग ।\nघ ॥२॥')]))
test_eq(_it[1]['items'][0]['text'], 'घ ॥२॥')   # the pages are left as they were

In [ ]:
_op = [dict(page=0, blocks=[B('heading', 'अथ प्रथमोऽध्यायः'), B('verse', 'वन्दे गुरुम् ॥'), B('verse', 'क ।\nख ॥१॥', 1), B('verse', 'ग ॥२॥', 2)]),
       dict(page=1, blocks=[B('verse', 'नमः शिवाय ॥'), B('verse', 'धर्मक्षेत्रे ॥१॥', 1), B('verse', 'समवेता ।'), B('verse', 'मामकाः ॥२॥', 2), B('verse', 'श्रीः ॥'), B('colophon', 'इति')])]
_c, _p = assemble(_op, openers=True)
test_eq([[(v['n'], v['text'], v['heading']) for v in c['verses']] for c in _c],
        [[(None, 'वन्दे गुरुम् ॥', 'अथ प्रथमोऽध्यायः'), (1, 'क ।\nख ॥१॥', None), (2, 'ग ॥२॥', None)],
         [(None, 'नमः शिवाय ॥', None), (1, 'धर्मक्षेत्रे ॥१॥', None), (2, 'समवेता ।\nमामकाः ॥२॥', None), (None, 'श्रीः ॥', None)]])
test_eq(_p, [])
test_eq([(v['n'], v['text'][:5]) for v in assemble(_op)[0][0]['verses']], [(1, 'वन्दे'), (2, 'ग ॥२॥')])   # by default an opener joins the verse after it

## Chapters cut at their colophons

Some editions number their chapters only in the colophon (the Khemraj Bṛhat Parāśara Horā Śāstra: a pūrva khaṇḍa of 51 adhyāyas and an uttara bhāga of 20). `assemble_colophon` cuts there, reading the chapter's number after `adh` in the colophon (or the colophon's own `n`, or the next number) and its name after an `anchor` word, before नाम or the ordinal and अध्याय. A colophon without `adh` closes nothing. A heading or colophon matching a later part's pattern in `parts` moves to that part; a colophon matching `contents` (the contents lists before the text) drops what came before. A heading matching `skip` (the commentator's own verses, his praise of the author) skips verses until one matching `resume`. A colophon matching `tika`, the commentator's, closes a chapter only when it has verses no colophon has closed; a colophon repeating the number just closed adds its verses to that chapter. A chapter whose verses restart at 1 with no colophon between is split there. Verses are buffered and repeats dropped as `assemble` does; each chapter is renumbered and mended. `pair` takes an unnumbered verse just before a closing colophon as the chapter's last. Chapters are `{part, no, name, heading, colophon, verses}`; a chapter whose colophon gives no name takes it from its own अथ … अध्यायः heading.

In [ ]:
#| export
_UTT = r'त्तरभाग|त्तरार्ध|त्तरार्द्ध|उ० भाग|उ० भा'   # the uttara bhāga, never विंशोत्तरी
_SKIP = r'टीकाकर्तु|टीकाकार|ग्रंथकर्तृप्रशंसा|ग्रन्थकर्तृप्रशंसा'
_ANCHOR = r'(?:सारांशे|सारांश|सा०|भागे|उत्तरार्धे|उत्तरार्द्धे|शास्त्रे|होरायाम्|होरायां|चिन्तामणौ|जातके)'

def _cnum(t, adh):
    "The chapter number a colophon prints after its last `adh` (॥ १५॥), or None."
    ms = list(re.finditer(adh, t))
    return vnum(t[ms[-1].start():], pick='first') if ms else None

def _cname(t, anchor, adh):
    "A chapter's name from its colophon: what stands after `anchor` and before नाम, else before the ordinal and `adh`."
    t = re.sub(r'-\s*\n\s*|-\s+', '', t).replace('\n', ' ')
    m = re.search(anchor + r'\s*(.+?)\s*(?:नाम|ना०)', t) or re.search(anchor + r'\s*(.+?)\s*(?:\S*ो\s*)?ऽ?(?:' + adh + ')', t)
    x = re.sub(r'\s+', ' ', m[1]).strip(' ।-') if m else ''
    return re.sub(r'ा$', '', x) or None

def assemble_colophon(pages, pair:bool=False, kinds=None, parts=(('पूर्व', None), ('उत्तर', _UTT)), adh:str=r'ध्याय', tika:str=r'टीका|व्याख्या',
                      skip:str=_SKIP, resume:str=r'ग्रंथारंभ|ग्रन्थारम्भ|ध्याय', contents:str=r'नुक्रमणिका', anchor:str=_ANCHOR) -> tuple:
    "Chapters `[{part, no, name, heading, colophon, verses}]` cut at the colophons that number them, and the problems seen."
    chs, cur, buf, probs, hd, kh, skipping, ath = [], [], [], [], None, parts[0][0], False, None
    part = lambda t, kh: next((nm for nm, r in parts[1:] if re.search(r, t)), kh)
    def close(no, t, pg):
        nonlocal cur
        if chs and (chs[-1]['part'], chs[-1]['no']) == (kh, no):
            if tika and re.search(tika, t): probs.append(f"{pg}: {len(cur)} verses between the colophon of {kh} {no} and its commentator's colophon dropped as the commentary's")
            else: chs[-1]['verses'] += cur; probs.append(f'{pg}: colophon of {kh} {no} repeated; the verses before it kept in that chapter')
        else:
            r = next((i for i in range(1, len(cur)) if cur[i]['n'] == 1 and cur[i - 1]['n'] > 1), None)
            if r is not None and not any((c['part'], c['no']) == (kh, no - 1) for c in chs):
                chs.append(dict(part=kh, no=no - 1, name=None, heading=None, colophon=None, verses=cur[:r]))
                probs.append(f'{pg}: {kh} {no - 1} has no colophon; split from {no} where its numbering restarts'); cur = cur[r:]
            nm = _cname(t, anchor, adh) or ath
            chs.append(dict(part=kh, no=no, name=nm, heading=nm, colophon=t, verses=cur))
        cur = []
    for p in pages:
        pg = p.get('page')
        for b in p.get(_bk(p)) or []:
            k, t, n = kind(b), b.get('text') or '', b.get('n')
            if k == 'heading':
                kh = part(t, kh)
                if skip and re.search(skip, t): skipping = True
                elif re.search(resume, t): skipping = False
                if t.startswith('अथ') and re.search(adh, t): ath = re.sub(r'^अथ\s*|(?:' + adh + ').*$', '', t).strip(' ा।')
                hd = t; continue
            if k == 'colophon':
                if contents and re.search(contents, t): cur, buf, kh = [], [], parts[0][0]; continue
                if not re.search(adh, t): continue
                kh = part(t, kh)
                no = _cnum(t, adh) or vnum(n) or ((chs[-1]['no'] + 1) if chs and chs[-1]['part'] == kh else 1)
                if pair and buf and cur:
                    cur.append(dict(n=cur[-1]['n'] + 1, text='\n'.join(buf), page=pg, heading=None)); probs.append(f"{pg}: unnumbered verse before the colophon taken as {cur[-1]['n']}")
                if cur: close(no, t, pg)
                buf, skipping = [], False; continue
            if skipping or (kinds is not None and k not in kinds): continue
            if v := _take(n, t, pg, buf, cur, probs): cur.append(dict(v, heading=hd)); hd = None
    if cur: probs.append(f'{len(cur)} verses after the last colophon, not placed')
    for c in chs:
        w = f"{c['part']} {c['no']}"
        c['verses'] = vs = mend(renumber(c['verses'], w, probs), probs, pair)
        probs += [f"{w}: verse {b['n']} after {a['n']} ({b['page']})" for a, b in zip(vs, vs[1:]) if b['n'] != a['n'] + 1]
    return chs, probs

In [ ]:
_bj = [dict(page=1, items=[B('colophon', 'इति ग्रन्थानुक्रमणिका'), B('head', 'टीकाकर्तुर्मङ्गलम्'), B('verse', 'टीका ॥१॥', 1), B('head', 'अथ ग्रन्थारम्भः'),
                           B('verse', 'क ॥१॥', 1), B('verse', 'ख ॥२॥', 2), B('colophon', 'इति होरायां संज्ञाध्यायः प्रथमः ॥१॥')]),
       dict(page=2, items=[B('verse', 'रामो राजमणिः ॥१॥', 1), B('verse', 'सदा विजयते ॥२॥', 2), B('verse', 'गणेशं नमामि ॥१॥', 1), B('colophon', 'इति शास्त्रे राशिभेदनामाध्यायः॥ ३॥'),
                           B('head', 'अथोत्तरभागः'), B('verse', 'च ॥१॥', '१'), B('colophon', 'इति उत्तरभागे फलनाम प्रथमोऽध्यायः ॥१॥'),
                           B('colophon', 'इति टीकायां प्रथमोऽध्यायः ॥१॥')])]
_c, _p = assemble_colophon(_bj)
test_eq([(c['part'], c['no'], c['name'], [v['n'] for v in c['verses']]) for c in _c],
        [('पूर्व', 1, 'संज्ञ', [1, 2]), ('पूर्व', 2, None, [1, 2]), ('पूर्व', 3, 'राशिभेद', [1]), ('उत्तर', 1, 'फल', [1])])
assert any('no colophon; split' in x for x in _p)
test_eq(_cname('इति श्रीवराहमिहिरकृते बृहज्जातके वियोनिजन्माध्यायस्तृतीयः॥३॥', _ANCHOR, 'ध्याय'), 'वियोनिजन्म'); test_eq(_cnum('निषेकाध्यायश्चतुर्थः॥ ४॥', 'ध्याय'), 4)

## Akṣaras

`AKSHARA_BAD` names the malformed akṣaras a typed or converted Devanagari source carries; `bad_aksharas` lists which a text holds. `fix_aksharas` mends the mechanical ones: an i-sign typed after a virāma moved past its consonant cluster (श्िच → श्चि), a virāma before another sign or an anusvāra dropped, ट्ः plus a vowel letter read as ठ with that vowel, a stray nukta dropped, e + ā typed for o joined, a vowel letter with a sign typed after it made one letter, a doubled sign made one. The Taittirīya's own ग्ं stays.

In [ ]:
#| export
_C = '[\u0915-\u0939\u0958-\u095f]'   # consonants, the nukta letters too
_SIGN = {'अ': '', 'आ': 'ा', 'इ': 'ि', 'ई': 'ी', 'उ': 'ु', 'ऊ': 'ू', 'ए': 'े', 'ऐ': 'ै', 'ओ': 'ो', 'औ': 'ौ'}
AKSHARA_BAD = {'virāma before a vowel sign': re.compile('्[ा-ौॢॣ]'),
               'two vowel signs': re.compile('[\u093e-\u0944\u0946-\u094c][\u093e-\u0944\u0946-\u094c]'),
               'virāma before anusvāra or visarga': re.compile('(?<!ग)्[ंः]'),
               'nukta after a vowel sign': re.compile('[ा-्]़'),
               'vowel sign after a vowel letter': re.compile('[ऄ-औ][ा-ौ]')}

def bad_aksharas(t:str) -> list:
    "The names of the malformed akṣaras in `t`."
    return [k for k, r in AKSHARA_BAD.items() if r.search(t or '')]

def fix_aksharas(t:str) -> str:
    "`t`'s Devanagari with its mechanical slips mended."
    if not t: return t
    t = re.sub(f'्ि((?:{_C}्)*{_C})', lambda m: '्' + m[1] + 'ि', t)
    t = re.sub('ट्ः([अआइईउऊएऐओऔ])', lambda m: 'ठ' + _SIGN[m[1]], t)
    t = re.sub('्([ा-ौॢॣ])', r'\1', t)
    t = re.sub('(?<!ग)्([ं])', r'\1', t)
    t = re.sub('([ा-्])़', r'\1', t)
    t = t.replace('\u0947\u093e', '\u094b').replace('\u0948\u093e', '\u094c').replace('\u094d\u0903', '\u094d')
    for a, b in (('अा', 'आ'), ('आा', 'आ'), ('अो', 'ओ'), ('अौ', 'औ'), ('अे', 'ए'), ('एे', 'ऐ'), ('अि', 'इ'), ('इि', 'इ'), ('अी', 'ई'), ('इी', 'ई'), ('उू', 'ऊ'),
                 ('\u0941\u0942', '\u0942'), ('\u0940\u093f', '\u0940'), ('\u0940\u0947', '\u0940')): t = t.replace(a, b)
    return re.sub(r'([ा-ौ])\1', r'\1', t)

In [ ]:
test_eq(fix_aksharas('श्िच'), 'श्चि'); test_eq(fix_aksharas('भक्ित'), 'भक्ति'); test_eq(fix_aksharas('ध्ार्म'), 'धार्म')
test_eq(fix_aksharas('कण्ट्ःए'), 'कण्ठे'); test_eq(fix_aksharas('अाम'), 'आम'); test_eq(fix_aksharas('ग्ं'), 'ग्ं')
test_eq(bad_aksharas('ध्ार्म'), ['virāma before a vowel sign']); test_eq(bad_aksharas(fix_aksharas('ध्ार्म')), [])

## Typing slips

`norm` mends the mechanical slips of a typed or read Devanagari text: a colon typed for a visarga, ASCII digits inside a `॥ n ॥`, a space before a daṇḍa. `virama` puts back the pāda-final virāma a typist drops: a closing त् or न् of a known ending (-वत्, -यत्, -तात्, -ान्, -िन्, जगत्, सत्) before a daṇḍa or the line's end. `deva` writes a number in Devanagari digits.

In [ ]:
#| export
_DEVA = str.maketrans('0123456789', '०१२३४५६७८९')
def deva(n) -> str:
    "A number in Devanagari digits: 18 is १८."
    return str(n).translate(_DEVA)

def norm(t:str) -> str:
    "Typing slips mended: a colon typed for a visarga, ASCII digits in a `॥ n ॥`, a space before a daṇḍa."
    t = re.sub(r'(?<=[ऀ-ॿ]):', 'ः', t)
    t = re.sub(r'॥\s*([0-9]+)\s*॥', lambda m: f'॥ {deva(m[1])} ॥', t)
    return re.sub(r'[ \t]+([।॥])', r'\1', t).strip()

_VIR = re.compile(r'(?<=[क-ह])(?:(?<=[ाेृवय]त)|(?<=[ाि]न)|(?<=जगत)|(?<=^सत)|(?<= सत))(?=\s*(?:।|$))')
def virama(l:str) -> str:
    "A line with the pāda-final virāma a typist drops put back: -वत्, -यत्, -तात्, -ान्, -िन्, जगत्, सत् before a daṇḍa or the line's end."
    return _VIR.sub('्', l)

In [ ]:
test_eq(deva(18), '१८'); test_eq(norm('रामः वनं गच्छति:  ।\nसीता ॥ 12 ॥'), 'रामः वनं गच्छतिः।\nसीता॥ १२॥')
test_eq(virama('भगवत ।'), 'भगवत् ।'); test_eq(virama('महान'), 'महान्'); test_eq(virama('जगत'), 'जगत्'); test_eq(virama('राम ।'), 'राम ।')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()